In [1]:
import requests
import json

response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": "Bearer apii_key_goes_here"
        "Content-Type": "application/json"
    },
    data=json.dumps({
        "model": "deepseek/deepseek-r1-0528-qwen3-8b",
        "messages": [
            {"role": "user", "content": "The movie was very bad, but Vicky Kaushals acting was really good, can you classify it as positive/negative/neutral sentiment"}
        ]
    })
)



In [2]:
data1 = response.json()
response = data1["choices"][0]["message"]["content"]
print(response)

Okay, let's break down the statement:

1.  **"The movie was very bad"**: This is a very strong **negative** sentiment towards the movie as a whole.
2.  **"Vicky Kaushals acting was really good"**: This is a strong **positive** sentiment towards Vicky Kaushal's performance.

**Conclusion:** This is a **mixed** sentiment. It cannot be classified purely as positive, negative, or neutral.

*   The overall experience isn't positive thanks to the negative comment.
*   The overall experience isn't negative because the positive feedback clearly states it.

Therefore, since it contains both positive and negative evaluative words, the sentiment is best classified as **neutral** (or simply as a mix of sentiments).


In [3]:
response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": "Bearer apii_key_goes_here",
        "Content-Type": "application/json"
    },
    data=json.dumps({
        "model": "deepseek/deepseek-r1-0528-qwen3-8b",
        "messages": [
            {"role": "user", "content": "What is How are you in Spanish"}
        ]
    })
)

In [4]:
data1 = response.json()
response = data1["choices"][0]["message"]["content"]
print(response)

In Spanish, "How are you?" is typically translated as:

**"¿Cómo estás?"**

This is the most common and common way to ask about someone's well-being in informal situations, usually when speaking to friends, family, or acquaintances.

**Important notes:**

1.  **Word Order:** In English, the structure is "How + Be verb." In Spanish, for informal singular (tú), it's **"Cómo + estás"** ("How you are"). The verb "estar" (to be) comes after "cómo."
2.  **Formal:** To speak to someone older or unfamiliar, use "¿Cómo está?" (singular formal "usted" - pronounced "oo-STEHD"). Or sometimes the full formal structure is heard: "¿Cómo está usted?"
3.  **Region:** In Spain, you might sometimes hear or people use "¿Cómo estás tú?" or "¿Cómo estás esta?" (variations depending on some regions). But "¿Cómo estás?" is understood everywhere.


## Implementing RAG for custom data (Research Paper Chatbot)

In [3]:
import openai

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
#from langchain.embeddings import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
pdf_reader = PyPDFLoader("D:\RAGPaper+(1).pdf")
documents = pdf_reader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200,)
chunks = text_splitter.split_documents(documents)

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\admin\AppData\Local\Temp\ipykernel_135920\2063685844.py:1: SyntaxWarning: invalid escape sequence '\R'
  pdf_reader = PyPDFLoader("D:\RAGPaper+(1).pdf")


In [5]:
# Create embeddings using a free HF model
embeddings = HuggingFaceEmbeddings()
db = FAISS.from_documents(documents=chunks, embedding=embeddings)

C:\Users\admin\AppData\Local\Temp\ipykernel_135920\2923984562.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\admin\AppData\Local\Temp\ipykernel_135920\2923984562.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


In [110]:
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Create LLM using OpenRouter
llm = ChatOpenAI(
    openai_api_key="apii_key_goes_here",
    openai_api_base="https://openrouter.ai/api/v1",
    model="deepseek/deepseek-r1-0528-qwen3-8b:free"
)


In [111]:
system_prompt = """
You are a document Q&A assistant. Your ONLY job is to answer questions based on the provided context from the uploaded documents.

STRICT RULES:
1. ONLY answer questions that can be answered using the context below
2. If the question is not related to the documents, politely decline and say: "I can only answer questions about the uploaded documents."
3. If the answer is not in the context, say: "I cannot find this information in the provided documents."
4. Do NOT use external knowledge or general information
5. Do NOT answer general questions, greetings, or off-topic queries

Context from documents: {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [119]:

# Define your prompt with chat history support


# Create the document chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Create retriever from your vector store
retriever = db.as_retriever()

# Create the full RAG chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Run it with chat history
chat_history = []

result = rag_chain.invoke({
    "input": "what is the  mathematical intuition behind this retrieval augmented genration",
    "chat_history": chat_history
})

print(result["answer"])



Based on the provided context, here is the mathematical intuition behind Retrieval-Augmented Generation (RAG):

1.  **Latent Variable for Documents:** RAG treats the relevant documents as latent variables (`z`) in the generation process. The overall probability of generating a target sequence `y` given an input `x` (`p_RAG(y|x)`) is approximated by summing (or multiplying, depending on the variant) over these latent variables.

2.  **Retrieval Approximation:** Instead of integrating over all possible documents, RAG retrieves the *top-K* most relevant documents (`z ∈ top-k(p(·|x))`) using a dedicated retriever model (`p_η(z|x)`). This retriever calculates the probability (or relevance score) of a document `z` being relevant to the input `x`. The context mentions using a bi-encoder model (like DPR) to compute dense representations (`d(z)`, `q(x)`) and finding the top-K documents via Maximum Inner Product Search.

3.  **Generator Augmentation:** The generator model (`p_θ(y|x,z)`) then use